# Detailed Notebook 4: Miscellaneous Functions, Usage Patterns, and Physics Scenarios

This notebook is a curated reference of the most frequently used Python/NumPy/SciPy
tools in computational physics.  For each tool you will find:

- **What it does** conceptually
- **How to call it** (syntax)
- **A worked physics example** showing why it matters
- **Common pitfalls** to avoid

**Contents**
1. Lambda functions — what they are and how to use them
2. NumPy array creation tools: `linspace`, `arange`, `array`
3. Array math and broadcasting — why NumPy beats plain Python lists
4. Conditional selection: `np.where`, `np.nonzero`
5. Linear algebra: `np.dot`, `np.cross`, `np.linalg.norm`
6. Plotting essentials: `plot`, `scatter`, `hist`, `imshow`
7. Common physics constant usage with `scipy.constants`
8. Scenario: projectile with velocity-dependent drag
9. Scenario: electric field from Coulomb's law
10. Scenario: Monte Carlo methods
11. Scenario: histogram and counting statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import epsilon_0, mu_0, c, g
from numpy.linalg import norm

---
## Section 1: Lambda functions

### What is a lambda?

A `lambda` is a **one-line anonymous function**.  It is identical to a `def`
function in every way that matters, just more compact.

```python
# These two are exactly equivalent:
f = lambda x: np.sin(x)

def f(x):
    return np.sin(x)
```

### Syntax

```
lambda arg1, arg2, ...: expression_that_is_returned
```

There is no `return` keyword — the expression after `:` is always returned.

### Multiple arguments

```python
area = lambda base, height: 0.5 * base * height
area(3, 4)   # → 6.0

# Three arguments
rhs = lambda t, y, k: -k * y    # for an ODE rhs
```

### Why lambdas appear everywhere in physics code

Functions like `quad`, `root_scalar`, `solve_ivp`, `curve_fit`, and `minimize`
all require a **callable** — something you can call like a function.
When the function is simple, a lambda is cleaner than defining a full `def`.

```python
# Without lambda:
def my_func(x):
    return np.cos(x) - x
root_scalar(my_func, bracket=[0, 1])

# With lambda (same result, one line):
root_scalar(lambda x: np.cos(x) - x, bracket=[0, 1])
```

### When NOT to use a lambda

- When the function needs more than one line of logic → use `def`.
- When you want to add a docstring → use `def`.
- When readability matters more than brevity → use `def`.

In [ ]:
# ── Demonstration 1a: lambda vs def ─────────────────────────────────────────
# Both produce identical callables.

f_def = None
def f_def(x):
    return x**2 + 2*x - 3

f_lam = lambda x: x**2 + 2*x - 3

x_test = np.linspace(-4, 3, 7)
print('def   result:', f_def(x_test))
print('lambda result:', f_lam(x_test))
print('Identical?', np.allclose(f_def(x_test), f_lam(x_test)))

In [ ]:
# ── Demonstration 1b: lambda capturing a variable from outer scope ───────────
# Lambdas can "capture" variables defined outside them.

decay_constant = 0.7   # defined in the outer scope

# This lambda captures decay_constant from the surrounding code.
# When we call rhs(t, y), it uses whatever decay_constant is at call time.
rhs = lambda t, y: -decay_constant * y

print('rhs(0, 100) =', rhs(0, 100))   # -0.7 * 100 = -70.0

# This is exactly how the decay ODE is typically written for solve_ivp.

In [ ]:
# ── Demonstration 1c: lambda inside a function call ──────────────────────────
from scipy.integrate import quad
from scipy.optimize import root_scalar

# quad expects: f(x) → float
I, _ = quad(lambda x: np.exp(-x**2), 0, 3)
print(f'∫₀³ exp(-x²) dx = {I:.6f}')

# root_scalar expects: f(x) → float
sol = root_scalar(lambda x: np.cos(x) - x, bracket=[0, 1])
print(f'Fixed point of cos(x)=x: {sol.root:.6f}')

**Key takeaway on lambdas:**
- `lambda x: expr` is a nameless function of one variable.
- Multiple arguments: `lambda x, y, z: expr`.
- They can reference variables from the surrounding scope.
- Use them when the function is simple and you want compact code.
- For anything complex, use `def` for readability and debuggability.

---
## Section 2: NumPy array creation tools

### `np.linspace(start, stop, N)`
Creates **N evenly spaced numbers** from `start` to `stop` **inclusive**.
Use this when you know how many points you want.

```python
np.linspace(0, 2*np.pi, 100)   # 100 points from 0 to 2π
```

### `np.arange(start, stop, step)`
Creates values from `start` to `stop` (exclusive) with a fixed `step`.
Use when you know the step size, not the number of points.

```python
np.arange(0, 10, 0.5)   # 0.0, 0.5, 1.0, ..., 9.5
```

### `np.array([...])`
Converts a Python list to a NumPy array, enabling all the vectorised operations.

```python
x = np.array([1.0, 2.5, 4.0, 7.0])
```

### `np.zeros(N)` and `np.ones(N)`
Pre-allocate arrays of zeros or ones — useful for storing results in a loop.

In [ ]:
# ── Example 2: array creation and their differences ─────────────────────────

t_linspace = np.linspace(0, 1, 6)    # exactly 6 points, endpoints included
t_arange   = np.arange(0, 1, 0.2)    # step 0.2, endpoint NOT included

print('linspace(0, 1, 6):', t_linspace)
print('arange  (0, 1, 0.2):', t_arange)
print()

# Practical difference: always use linspace for plotting axes!
# arange can have floating-point step size issues for non-integer steps.
t_bad = np.arange(0, 1.0, 0.1)
print('arange(0,1,0.1) has', len(t_bad), 'elements  (can be 9 or 10 depending on float rounding)')
t_good = np.linspace(0, 1, 11)
print('linspace(0,1,11) has', len(t_good), 'elements  (always exactly 11)')

---
## Section 3: Array math and broadcasting

### Why NumPy arrays instead of lists?

With plain Python lists, arithmetic operates element-by-element only with loops.
NumPy arrays support **vectorised** (element-wise) operations — no loops needed.

```python
# Python list — won't work like this!
x = [1, 2, 3, 4]
y = x * 2      # → [1,2,3,4,1,2,3,4]  (list duplication, not multiplication!)

# NumPy array — works as expected
x = np.array([1, 2, 3, 4])
y = x * 2      # → [2, 4, 6, 8]
```

### Broadcasting

When two arrays have different shapes, NumPy tries to *broadcast* them:
align shapes and repeat along size-1 dimensions.

```python
x = np.array([1, 2, 3])   # shape (3,)
x + 10                     # broadcasts 10 to [10, 10, 10] → [11, 12, 13]
```

In [ ]:
# ── Example 3: vectorised physics — kinetic energy for many velocities ───────
masses = np.array([0.1, 0.5, 1.0, 2.0, 5.0])   # kg
v      = 10.0                                     # m/s (scalar broadcasts)

# Broadcasting: v**2 / 2 is a scalar, masses is an array → result is an array
KE = 0.5 * masses * v**2

print('mass (kg) | KE (J)')
print('-' * 25)
for m, ke in zip(masses, KE):
    print(f'{m:9.1f} | {ke:10.2f}')

# Vectorised sin, cos, exp — no loop needed
theta = np.linspace(0, 2*np.pi, 7)
print('\nsin(theta):', np.round(np.sin(theta), 3))

---
## Section 4: Conditional selection — `np.where` and `np.nonzero`

### `np.where(condition, x, y)`
Returns `x` where `condition` is True, `y` where False.
Think of it as a vectorised if-else.

```python
np.where(x > 0, x, 0)   # "clip to zero": return x if x>0, else 0
```

### `np.where(condition)` (one argument)
Returns the **indices** where condition is True.  Equivalent to `np.nonzero(condition)`.

```python
idx = np.where(y > 0.9)   # tuple of arrays of indices
y[idx]                     # the values that are > 0.9
```

### Common use cases in physics

- Find where a trajectory is above a threshold height.
- Select only data points with energy > some minimum.
- Apply different drag models below/above the speed of sound.

In [ ]:
# ── Example 4a: find peaks above a threshold ─────────────────────────────────
# Simulate a detector pulse train: Gaussian pulses at known times
t = np.linspace(0, 20, 2000)
pulse_times = [3.0, 7.5, 12.0, 16.5]
signal = sum(np.exp(-((t - pt)**2) / 0.1) for pt in pulse_times)
signal += 0.02 * np.random.default_rng(0).standard_normal(len(t))

threshold = 0.5

# Find indices where signal exceeds threshold
above = np.where(signal > threshold)
print(f'Fraction of samples above threshold: {len(above[0]) / len(t):.3f}')

# Time ranges where signal is above threshold (approximate)
t_above = t[above]
if len(t_above) > 0:
    print(f'First time above threshold: {t_above[0]:.3f} s')

plt.figure(figsize=(10, 3))
plt.plot(t, signal, linewidth=0.8, label='signal')
plt.axhline(threshold, color='r', linestyle='--', label=f'threshold = {threshold}')
plt.scatter(t[above], signal[above], s=1, color='red', zorder=5)
plt.xlabel('t (s)');  plt.ylabel('amplitude')
plt.title('np.where to select samples above threshold')
plt.legend();  plt.grid(True, alpha=0.3);  plt.show()

In [ ]:
# ── Example 4b: piecewise drag model using np.where ──────────────────────────
# Simple model: below Mach 0.8 use Cd=0.3, above use Cd=0.8 (transonic increase)

v_sound = 343.0   # m/s at sea level
v_range = np.linspace(0, 800, 500)   # m/s

Cd = np.where(v_range < 0.8 * v_sound, 0.3, 0.8)

plt.figure(figsize=(7, 3))
plt.plot(v_range, Cd)
plt.axvline(0.8*v_sound, color='r', linestyle='--', label='Mach 0.8')
plt.xlabel('Speed (m/s)');  plt.ylabel('Drag coefficient Cd')
plt.title('np.where: piecewise drag coefficient model')
plt.legend();  plt.grid(True, alpha=0.3);  plt.show()

---
## Section 5: Linear algebra — `np.dot`, `np.cross`, `np.linalg.norm`

These three functions are the workhorses of vector mechanics and field calculations.

### `np.dot(a, b)` — dot product / matrix multiply

For 1-D arrays: computes the scalar dot product a · b = Σ aᵢ bᵢ.
For 2-D arrays: computes matrix multiplication.

### `np.cross(a, b)` — cross product

Returns the vector perpendicular to both a and b with magnitude |a||b|sin(θ).

Physics uses: angular velocity, magnetic force (F = qv × B), torque (τ = r × F).

### `np.linalg.norm(v)` — vector magnitude

|v| = sqrt(v · v).  Use this to get distances, speeds, field magnitudes.

In [ ]:
# ── Example 5a: Lorentz force on a charged particle ──────────────────────────
# F = q * (v x B)
# v and B are 3-D vectors

q = 1.6e-19    # charge (Coulombs)
v = np.array([3e5, 0.0, 0.0])     # velocity in x-direction (m/s)
B = np.array([0.0, 0.0, 1.5])     # magnetic field in z-direction (Tesla)

F_lorentz = q * np.cross(v, B)
print('v =', v)
print('B =', B)
print('F = q(v×B) =', F_lorentz, 'N')
print('|F| =', norm(F_lorentz), 'N')

# Check: v ⊥ B means |F| = q|v||B|
F_magnitude_expected = q * norm(v) * norm(B)
print('Expected |F| = q|v||B| =', F_magnitude_expected, 'N')

In [ ]:
# ── Example 5b: dot product — work done by a force ───────────────────────────
# W = F · d   (force dotted with displacement)

F_force = np.array([3.0, -2.0, 1.0])   # Newtons
d_disp  = np.array([1.5,  0.5, 2.0])   # metres

W = np.dot(F_force, d_disp)
print(f'Work W = F·d = {W:.2f} J')

# Angle between F and d
cos_theta = np.dot(F_force, d_disp) / (norm(F_force) * norm(d_disp))
theta_deg = np.degrees(np.arccos(cos_theta))
print(f'Angle between F and d = {theta_deg:.2f} degrees')

In [ ]:
# ── Example 5c: angular momentum  L = r × p ──────────────────────────────────
m = 1.5   # kg
r = np.array([2.0, 1.0, 0.0])   # position (m)
v_vel = np.array([-1.0, 2.0, 0.0])  # velocity (m/s)

p = m * v_vel                  # momentum = m*v
L = np.cross(r, p)             # angular momentum
print('r =', r)
print('p =', p)
print('L = r × p =', L)
print('|L| =', norm(L), 'kg·m²/s')

---
## Section 6: Plotting essentials

A quick guide to the plot types used most in the class notes.

| Function | When to use |
|----------|------------|
| `plt.plot(x, y)` | Line plot — trajectories, functions, time series |
| `plt.scatter(x, y)` | Data points without connecting lines |
| `plt.hist(data)` | Distribution / counting statistics |
| `plt.imshow(Z)` | 2-D field map (colour plot of a matrix) |
| `plt.contourf(X, Y, Z)` | Smooth contour fill of a 2-D field |

In [ ]:
# ── Example 6: four plot types in one figure ─────────────────────────────────
rng = np.random.default_rng(42)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# 1. Line plot: damped oscillation
t = np.linspace(0, 10, 500)
y_damp = np.exp(-0.3*t) * np.cos(2*np.pi*t)
axes[0,0].plot(t, y_damp)
axes[0,0].set_title('plt.plot: damped oscillation')
axes[0,0].set_xlabel('t');  axes[0,0].set_ylabel('x(t)')
axes[0,0].grid(True, alpha=0.3)

# 2. Scatter plot: measurement data
x_pts = rng.uniform(0, 5, 50)
y_pts = 1.2*x_pts + rng.normal(0, 0.5, 50)
axes[0,1].scatter(x_pts, y_pts, s=20, alpha=0.7)
axes[0,1].set_title('plt.scatter: measurements')
axes[0,1].set_xlabel('x');  axes[0,1].set_ylabel('y')
axes[0,1].grid(True, alpha=0.3)

# 3. Histogram: statistical distribution
energies = rng.exponential(scale=2.0, size=2000)
axes[1,0].hist(energies, bins=40, edgecolor='k', linewidth=0.3)
axes[1,0].set_title('plt.hist: energy spectrum')
axes[1,0].set_xlabel('Energy (arb)');  axes[1,0].set_ylabel('counts')

# 4. imshow: 2-D field map
X_grid, Y_grid = np.mgrid[-3:3:100j, -3:3:100j]
Z_field = np.sin(X_grid) * np.cos(Y_grid)
im = axes[1,1].imshow(Z_field.T, origin='lower', extent=[-3,3,-3,3], cmap='RdBu')
plt.colorbar(im, ax=axes[1,1])
axes[1,1].set_title('plt.imshow: 2-D field')
axes[1,1].set_xlabel('x');  axes[1,1].set_ylabel('y')

plt.tight_layout()
plt.show()

---
## Section 7: Physical constants with `scipy.constants`

`scipy.constants` provides SI values for hundreds of constants.

```python
from scipy.constants import epsilon_0, mu_0, c, g, G, h, hbar, m_e, m_p, e, k
```

| Name | Symbol | Value |
|------|--------|-------|
| `epsilon_0` | ε₀ | 8.854e-12 F/m |
| `mu_0` | μ₀ | 4π×10⁻⁷ T·m/A |
| `c` | c | 2.998e8 m/s |
| `g` | g | 9.806 m/s² |
| `G` | G | 6.674e-11 m³/(kg·s²) |
| `h` | h | 6.626e-34 J·s |
| `m_e` | mₑ | 9.109e-31 kg |
| `e` | e | 1.602e-19 C |

In [ ]:
# ── Example 7: constants in calculations ────────────────────────────────────
from scipy.constants import epsilon_0, mu_0, c as c_light, G, h, m_e, e as q_e

# Bohr radius: a0 = 4*pi*eps0*hbar^2 / (m_e * e^2)
hbar = h / (2 * np.pi)
a0 = 4 * np.pi * epsilon_0 * hbar**2 / (m_e * q_e**2)
print(f'Bohr radius a₀ = {a0:.6e} m  (textbook: 5.292e-11 m)')

# Compton wavelength of the electron: lambda_c = h / (m_e * c)
lambda_c = h / (m_e * c_light)
print(f'Compton wavelength = {lambda_c:.6e} m  (textbook: 2.426e-12 m)')

# Fine structure constant: alpha = e^2 / (4*pi*eps0 * hbar * c)
alpha = q_e**2 / (4 * np.pi * epsilon_0 * hbar * c_light)
print(f'Fine structure constant α = {alpha:.8f}  (textbook: 1/137 ≈ 7.297e-3)')

---
## Section 8: Scenario — projectile with tabulated drag coefficient

### The physics

For a sphere moving through a fluid, the drag force depends on the Reynolds number:

    F_drag = -(1/2) * rho * Cd(Re) * A * v²  ×  v̂

where `Cd` depends on Re = ρ·v·D/μ and is typically given as a lookup table.

### The workflow

1. Load or define the `Cd(Re)` table.
2. Interpolate it to get `Cd` at any Reynolds number.
3. Define the ODE for the trajectory, calling the interpolant at each step.
4. Solve with `solve_ivp`.

In [ ]:
# ── Example 8: sphere falling with tabulated drag coefficient ─────────────────
from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp

# Tabulated Cd vs Re for a sphere (simplified)
Re_table = np.array([1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 4e5, 1e6])
Cd_table = np.array([24, 4.0, 1.1, 0.5, 0.40, 0.45, 0.10, 0.15])

Cd_of_Re = interp1d(Re_table, Cd_table, kind='linear',
                    fill_value=(Cd_table[0], Cd_table[-1]),
                    bounds_error=False)

# Physical parameters: steel sphere in air
D    = 0.01        # diameter 1 cm
rho_fluid = 1.22   # air density kg/m³
mu   = 1.81e-5     # dynamic viscosity of air
rho_sphere = 7800  # steel kg/m³
m    = rho_sphere * (4/3) * np.pi * (D/2)**3
A    = np.pi * (D/2)**2

def falling_sphere(t, state):
    y, vy = state   # position (upward positive), velocity
    speed = abs(vy)
    if speed < 1e-10:
        return [vy, -g]
    Re = rho_fluid * speed * D / mu
    Cd = float(Cd_of_Re(Re))
    F_drag = 0.5 * rho_fluid * Cd * A * speed**2
    sign_v = np.sign(vy)
    a = -g + sign_v * F_drag / m   # gravity down, drag opposing motion
    return [vy, a]

# Drop from 100 m at rest
sol_sphere = solve_ivp(falling_sphere,
                       t_span=[0, 5],
                       y0=[100.0, 0.0],
                       t_eval=np.linspace(0, 5, 1000),
                       rtol=1e-8, atol=1e-10)

# Terminal velocity = when acceleration = 0  → |vy| stabilises
vy = sol_sphere.y[1, :]
print(f'Speed at t=5 s: {abs(vy[-1]):.3f} m/s')
print(f'Analytic terminal velocity (constant Cd=0.47): '
      f'{np.sqrt(2*m*g / (rho_fluid*0.47*A)):.3f} m/s')

plt.figure(figsize=(7, 4))
plt.plot(sol_sphere.t, sol_sphere.y[0, :])
plt.xlabel('t (s)');  plt.ylabel('height (m)')
plt.title('Sphere falling with tabulated drag coefficient')
plt.grid(True, alpha=0.3);  plt.show()

---
## Section 9: Scenario — electric field from a point charge

### The physics

Coulomb's law gives the electric field from a point charge q at distance r:

    E = q / (4π ε₀ r²)

For a charge at a specific location, the field at (x, y) is a vector
pointing radially away from the charge.

### What we'll compute

- Field along a line through the charge
- 2-D field map showing magnitude

In [ ]:
# ── Example 9: Coulomb field map ─────────────────────────────────────────────
from scipy.constants import epsilon_0

q_charge = 1.6e-19   # proton charge (C)

# Grid of observation points
x_grid = np.linspace(-5e-10, 5e-10, 100)   # ±5 Angstrom
y_grid = np.linspace(-5e-10, 5e-10, 100)
X, Y = np.meshgrid(x_grid, y_grid)

r_sq = X**2 + Y**2
r_sq[r_sq < (5e-12)**2] = (5e-12)**2   # avoid singularity at origin

E_magnitude = q_charge / (4 * np.pi * epsilon_0 * r_sq)

plt.figure(figsize=(7, 6))
im = plt.imshow(np.log10(E_magnitude),
                origin='lower',
                extent=[x_grid.min()*1e10, x_grid.max()*1e10,
                        y_grid.min()*1e10, y_grid.max()*1e10],
                cmap='hot')
plt.colorbar(im, label='log₁₀|E| (V/m)')
plt.xlabel('x (Å)');  plt.ylabel('y (Å)')
plt.title('Electric field magnitude near a proton')
plt.scatter([0], [0], color='cyan', s=100, zorder=5, label='proton')
plt.legend();  plt.show()

# E along the x-axis for reference
r_line = np.linspace(5e-12, 5e-10, 200)
E_line = q_charge / (4 * np.pi * epsilon_0 * r_line**2)
print(f'E at 1 Å = {E_line[np.argmin(abs(r_line - 1e-10))]:.3e} V/m')

---
## Section 10: Scenario — Monte Carlo methods

### What is Monte Carlo?

Monte Carlo methods use **random sampling** to compute quantities that are
hard to calculate analytically.

**Physics uses:**
- Estimating multi-dimensional integrals
- Simulating radioactive decay chains
- Particle transport through matter
- Statistical mechanics (Metropolis algorithm)

### How it works for integration

To estimate ∫₀¹ f(x) dx:
1. Sample N random x values uniformly in [0, 1].
2. Evaluate f at each: f₁, f₂, …, fₙ.
3. Estimate: ∫f dx ≈ (1/N) Σ fᵢ

The law of large numbers guarantees convergence.  Error ∝ 1/√N.

In [ ]:
# ── Example 10a: Monte Carlo estimate of π ───────────────────────────────────
# The unit circle has area π.  The unit square has area 4.
# Sample random points in [-1,1]²; fraction inside unit circle ≈ π/4.

rng = np.random.default_rng(0)
results = []

for N in [100, 1000, 10000, 100000, 1000000]:
    pts = rng.uniform(-1, 1, size=(N, 2))
    inside = np.sum(pts[:, 0]**2 + pts[:, 1]**2 <= 1)
    pi_est = 4 * inside / N
    error = abs(pi_est - np.pi)
    results.append((N, pi_est, error))
    print(f'N={N:>8d}:  π ≈ {pi_est:.6f},  error = {error:.6f}')

print(f'\nTrue π = {np.pi:.6f}')
print('Note: error scales as 1/√N, so 100× more samples → 10× smaller error')

In [ ]:
# ── Example 10b: Monte Carlo integration in 2D ───────────────────────────────
# Estimate: ∫∫_{unit disk} e^(-r²) dx dy = π(1 - e^-1) ≈ 1.9862

rng2 = np.random.default_rng(7)
N2 = 500000
pts2 = rng2.uniform(-1, 1, size=(N2, 2))
r2 = pts2[:, 0]**2 + pts2[:, 1]**2

# Only accept points inside the unit disk
inside_disk = r2 <= 1
f_vals = np.exp(-r2[inside_disk])

# Area of unit disk × mean f value inside disk
mc_integral = np.pi * f_vals.mean()
exact = np.pi * (1 - np.exp(-1))
print(f'Monte Carlo 2D integral = {mc_integral:.6f}')
print(f'Exact value             = {exact:.6f}')
print(f'Relative error: {abs(mc_integral - exact)/exact:.4f}')

---
## Section 11: Scenario — histogram and counting statistics

### Physics context

Particle detectors record discrete counts.  The counts follow a Poisson
distribution when the process is random and uncorrelated.

**Poisson statistics:**
- Mean = μ
- Variance = μ (so standard deviation = √μ)
- For large μ the distribution approaches Gaussian with σ = √μ

`plt.hist` bins the data; the bin heights are the counts in each bin.
`np.mean`, `np.std` compute the sample mean and standard deviation.

In [ ]:
# ── Example 11: Poisson counting simulation ──────────────────────────────────
rng3 = np.random.default_rng(21)
true_rate = 50   # average counts per measurement interval

# Simulate 2000 repeated measurements
counts = rng3.poisson(lam=true_rate, size=2000)

sample_mean = counts.mean()
sample_std  = counts.std(ddof=1)
expected_std = np.sqrt(true_rate)

print(f'True rate: {true_rate}')
print(f'Sample mean: {sample_mean:.3f}   (should be ~{true_rate})')
print(f'Sample std:  {sample_std:.3f}    (should be ~{expected_std:.3f} for Poisson)')

# Compare histogram to theoretical Poisson distribution
from scipy.stats import poisson

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bins = np.arange(true_rate - 20, true_rate + 21, 1)
axes[0].hist(counts, bins=bins, density=True, edgecolor='k', linewidth=0.3,
             label='simulated counts')
k_range = np.arange(bins[0], bins[-1]+1)
axes[0].plot(k_range, poisson.pmf(k_range, true_rate), 'ro-', markersize=4,
             label='Poisson PMF')
axes[0].set_xlabel('count');  axes[0].set_ylabel('probability')
axes[0].set_title('Poisson count distribution')
axes[0].legend();  axes[0].grid(True, alpha=0.3)

# Running mean — converges to true rate as N increases
N_vals = np.arange(1, len(counts)+1)
running_mean = np.cumsum(counts) / N_vals
axes[1].semilogx(N_vals, running_mean, linewidth=0.8)
axes[1].axhline(true_rate, color='r', linestyle='--', label=f'true rate = {true_rate}')
axes[1].set_xlabel('number of measurements N (log scale)');  axes[1].set_ylabel('sample mean')
axes[1].set_title('Convergence of sample mean')
axes[1].legend();  axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Quick-reference summary

| Need | Tool |
|------|------|
| Anonymous one-line function | `lambda args: expression` |
| N evenly-spaced points | `np.linspace(a, b, N)` |
| Fixed step size | `np.arange(a, b, step)` |
| Element-wise math on arrays | Direct operators: `x**2`, `np.sin(x)`, etc. |
| Values where condition is true | `np.where(condition)` |
| Replace values conditionally | `np.where(cond, x_true, x_false)` |
| Dot product / matrix multiply | `np.dot(a, b)` |
| Cross product | `np.cross(a, b)` |
| Vector magnitude | `np.linalg.norm(v)` |
| Physical constants | `from scipy.constants import epsilon_0, G, ...` |